<a href="https://colab.research.google.com/github/akriti404/Deep-Learning-Lab/blob/main/Lab_Assessment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AKRITI AGARWAL - 23BDS0038

Question 1 -
Using the Text classification dataset (e.g., IMDb Movie Reviews or SST-2).
- Fine-tune a pre-trained bert-base-uncased model (using HuggingFace
Transformers) on a subset of the dataset (e.g., 2,000 samples).
- Extract and inspect the Attention Weights from the last multi-head attention layer
for two specific test sentences:
- Sentence A:The movie was not bad, in fact, it was quite enjoyable.
- Sentence B:The movie was bad, not enjoyable at all.
- Extract the pooled output of the [CLS] token embeddings for both sentences and
calculate the Cosine Similarity between their vector representations before fine-
tuning vs. after fine-tuning.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics.pairwise import cosine_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("\nLoading IMDb dataset...")
dataset = load_dataset("stanfordnlp/imdb")

train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

#Load Pre-trained BERT Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    attn_implementation="eager"
).to(device)

sentence_A = "The movie was not bad, in fact, it was quite enjoyable."
sentence_B = "The movie was bad, not enjoyable at all."

Using device: cuda

Loading IMDb dataset...


Training samples: 2000
Testing samples: 500


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
def get_cls_embedding(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.base_model(**inputs, return_dict=True)

    return outputs.pooler_output.cpu().numpy()

#Extract Embeddings BEFORE Fine-tuning
embedding_A_before = get_cls_embedding(sentence_A)
embedding_B_before = get_cls_embedding(sentence_B)

similarity_before = cosine_similarity(embedding_A_before, embedding_B_before)[0][0]

print("\nCosine Similarity BEFORE fine-tuning:",round(float(similarity_before), 4))


Cosine Similarity BEFORE fine-tuning: 0.9696


In [4]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_data = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
test_data = test_data.map(tokenize_function, batched=True, remove_columns=["text"])

columns = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in train_data.column_names:
    columns.insert(2, "token_type_ids")

train_data.set_format(type="torch", columns=columns)
test_data.set_format(type="torch", columns=columns)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [5]:
#Extract Last-Layer Attention Weights
def get_attention(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.base_model(
            **inputs,
            output_attentions=True,
            return_dict=True
        )
    attention = outputs.attentions[-1][0].cpu()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu().tolist())

    return tokens, attention

tokens_A, attention_A = get_attention(sentence_A)
tokens_B, attention_B = get_attention(sentence_B)

print("\nSentence A:", sentence_A)
print("Tokens:", tokens_A)
print("Attention shape:", tuple(attention_A.shape))

print("\nSentence B:", sentence_B)
print("Tokens:", tokens_B)
print("Attention shape:", tuple(attention_B.shape))


Sentence A: The movie was not bad, in fact, it was quite enjoyable.
Tokens: ['[CLS]', 'the', 'movie', 'was', 'not', 'bad', ',', 'in', 'fact', ',', 'it', 'was', 'quite', 'enjoyable', '.', '[SEP]']
Attention shape: (12, 16, 16)

Sentence B: The movie was bad, not enjoyable at all.
Tokens: ['[CLS]', 'the', 'movie', 'was', 'bad', ',', 'not', 'enjoyable', 'at', 'all', '.', '[SEP]']
Attention shape: (12, 12, 12)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], "f"Loss: {total_loss / len(train_loader):.4f}")

print("\nFine-tuning is done")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        predictions = outputs.logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

#Extract Embeddings AFTER Fine-tuning
embedding_A_after = get_cls_embedding(sentence_A)
embedding_B_after = get_cls_embedding(sentence_B)

similarity_after = cosine_similarity(embedding_A_after, embedding_B_after)[0][0]

print("\nCLS Cosine Similarity Comparison")
print(f"Before fine-tuning: {similarity_before:.4f}")
print(f"After fine-tuning : {similarity_after:.4f}")

Epoch [1/2], Loss: 0.1191
Epoch [2/2], Loss: 0.0488

Fine-tuning is done
Test Accuracy: 85.20%

CLS Cosine Similarity Comparison
Before fine-tuning: 0.9696
After fine-tuning : -0.8685
